In [1]:
import warnings

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [4]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=ColourContext  
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='1b667ae5-f358-4b6e-9347-dfb22b8d2617'),
              AIMessage(content="I don't have access to personal information like your favorite color! Could you share it with me? I'd love to know! 🌈", additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-03-10T20:47:48.005376555Z', 'done': True, 'done_reason': 'stop', 'total_duration': 132959608132, 'load_duration': 30480052294, 'prompt_eval_count': 16, 'prompt_eval_duration': 4199230736, 'eval_count': 218, 'eval_duration': 98151287061, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019cd97f-75c1-76a0-9615-3d1f2271363b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 218, 'total_tokens': 234})]}


In [8]:
print(response["messages"][-1].content)

I don't have access to personal information like your favorite color! Could you share it with me? I'd love to know! 🌈


## Accessing Context

In [9]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [10]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='ddeb86c5-8879-42cd-be4d-3714f79c316d'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-03-10T20:51:24.366437826Z', 'done': True, 'done_reason': 'stop', 'total_duration': 96157905705, 'load_duration': 145316536, 'prompt_eval_count': 166, 'prompt_eval_duration': 35942803850, 'eval_count': 128, 'eval_duration': 59990275049, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019cd983-52ab-7c31-abc3-0008fcb8c642-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'f8e83395-85aa-43d5-b54f-e11fa0a9a5c2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 128, 'total_tokens': 294}),
              ToolMessage(content='blue', name='get_favourite_colour', id='c2a7b0e2-a135-41bd-a717-77b516661c75', tool_call_id='f8e8

In [12]:
print(response["messages"][-1].content)

Your favourite colour is **blue**. Let me know if you'd like to explore anything else! 😊


In [13]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='06139ca4-985d-4117-8e04-c04b3fc25b02'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:14b', 'created_at': '2026-03-10T20:54:31.21915817Z', 'done': True, 'done_reason': 'stop', 'total_duration': 58035142131, 'load_duration': 151877700, 'prompt_eval_count': 166, 'prompt_eval_duration': 428090319, 'eval_count': 128, 'eval_duration': 57393080165, 'logprobs': None, 'model_name': 'qwen3:14b', 'model_provider': 'ollama'}, id='lc_run--019cd986-c17e-7461-a6bf-43142c93611c-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'dd8b6073-90b0-4f1c-8a04-be864084d020', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 128, 'total_tokens': 294}),
              ToolMessage(content='green', name='get_favourite_colour', id='12517f1f-eeb6-4ce7-a68b-47473fea599e', tool_call_id='dd8b60

In [14]:
print(response["messages"][-1].content)

Your favourite colour is **green**. Let me know if you need help with anything else! 🌿
